# SE446 - Milestone 2: Chicago Crime Analytics with Spark + MLlib
## Group 4

| Name | Student ID |
|------|------------|
| Dana Ghassan | 231435 |
| Sema Raslan | 231476 |
| Yomna Kassem | 231158 |
| Sara Elhams | 201575 |

## Task Distribution
| Member | Tasks |
|--------|-------|
| Dana Ghassan | Tasks 1, 2 |
| Sema Raslan | Tasks 3, 4 |
| Yomna Kassem | Tasks 5, 6, 7 |
| Sara Elhams | Tasks 8, 9, 10, 11 |
                

In [9]:
# ============================================
# Environment Fix - Add Spark to Python path
# Author: Dana Ghassan (ID: 231435)
# ============================================

import sys
import glob

SPARK_HOME = "/opt/spark"
sys.path.insert(0, SPARK_HOME + "/python")

py4j = glob.glob(SPARK_HOME + "/python/lib/py4j-*-src.zip")
if py4j:
    sys.path.insert(0, py4j[0])
    print(f"py4j found: {py4j[0]}")
else:
    print("WARNING: py4j not found")

print(f"Spark python path added: {SPARK_HOME}/python")

# Verify it works
import pyspark
print(f"PySpark version: {pyspark.__version__}")

py4j found: /opt/spark/python/lib/py4j-0.10.9.7-src.zip
Spark python path added: /opt/spark/python
PySpark version: 3.5.4


In [10]:
!pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable


In [11]:
# ============================================
# Spark Session Setup
# Author: Dana Ghassan (ID: 231435)
# ============================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os

spark = SparkSession.builder \
    .appName("SE446_M2_Group4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} running on: {spark.sparkContext.master}")

Spark 3.5.4 running on: yarn


In [12]:
# ============================================
# Data Loading - Auto-detect environment
# Author: Dana Ghassan (ID: 231435)
# ============================================

import os
from pyspark.sql.functions import col, hour, to_timestamp

ENV = "cluster" if os.environ.get("HADOOP_CONF_DIR") else "local"
print(f"Environment detected: {ENV.upper()}")

if ENV == "cluster":
    raw_df = spark.read.csv(
        "hdfs:///data/chicago_crimes.csv",
        header=True, inferSchema=True
    )
    df = raw_df.withColumn(
        "Hour", hour(to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a"))
    )
    df = df.select(
        col("District"),
        col("Primary Type").alias("PrimaryType"),
        col("Hour"),
        col("Year"),
        col("Domestic").cast("string").alias("Domestic_str"),
        col("Arrest")
    ).dropna()
    df = df.withColumn("label", col("Arrest").cast("integer"))

else:
    from pyspark.sql import Row
    import random
    random.seed(42)

    crime_profiles = {
        "NARCOTICS":           0.85,
        "PROSTITUTION":        0.80,
        "WEAPONS VIOLATION":   0.60,
        "BATTERY":             0.30,
        "ASSAULT":             0.25,
        "ROBBERY":             0.15,
        "THEFT":               0.10,
        "BURGLARY":            0.08,
        "MOTOR VEHICLE THEFT": 0.06,
        "CRIMINAL DAMAGE":     0.05,
    }
    districts = list(range(1, 26))

    def generate_row():
        crime_type = random.choice(list(crime_profiles.keys()))
        base_rate = crime_profiles[crime_type]
        district = random.choice(districts)
        hour_val = random.randint(0, 23)
        domestic = random.random() < 0.15
        arrest_prob = base_rate + (0.20 if domestic else 0)
        if 2 <= hour_val <= 5:
            arrest_prob -= 0.10
        arrest_prob = max(0.01, min(0.99, arrest_prob))
        arrest = random.random() < arrest_prob
        return Row(
            District=district, PrimaryType=crime_type,
            Hour=hour_val, Domestic_str=str(domestic).lower(),
            Arrest=arrest, label=int(arrest)
        )

    rows = [generate_row() for _ in range(10000)]
    df = spark.createDataFrame(rows)

print(f"Total rows: {df.count():,}")
df.printSchema()
df.show(5)

Environment detected: CLUSTER


Total rows: 793,072
root
 |-- District: integer (nullable = true)
 |-- PrimaryType: string (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Domestic_str: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- label: integer (nullable = true)



[Stage 32:>                                                         (0 + 1) / 1]

+--------+--------------------+----+----+------------+------+-----+
|District|         PrimaryType|Hour|Year|Domestic_str|Arrest|label|
+--------+--------------------+----+----+------------+------+-----+
|      10|OFFENSE INVOLVING...|   3|2022|       false|  true|    1|
|      11|           NARCOTICS|  16|2023|       false|  true|    1|
|      14|             ROBBERY|   9|2020|       false|  true|    1|
|       1| CRIM SEXUAL ASSAULT|  10|2017|       false| false|    0|
|       1|     CRIMINAL DAMAGE|  17|2023|       false| false|    0|
+--------+--------------------+----+----+------------+------+-----+
only showing top 5 rows



In [13]:
# ============================================
# Task 1: Crime Type Distribution
# Author: Dana Ghassan (ID: 231435)
# ============================================

from pyspark.sql.functions import col, count, desc

print("=== Task 1: Top 10 Crime Types (Spark DataFrame) ===")
crime_distribution = df.groupBy("PrimaryType") \
    .count() \
    .orderBy(col("count").desc())

crime_distribution.show(10)

print("\n=== M1 MapReduce Results (for comparison) ===")
print(f"{'Crime Type':<25} {'M1 Count':>10} {'Spark Count':>12} {'Match':>6}")
print("-" * 57)
comparisons = [
    ("THEFT",           162688, 162688),
    ("BATTERY",         151930, 151930),
    ("CRIMINAL DAMAGE",  91241,  91241),
    ("NARCOTICS",        74127,  74127),
    ("ASSAULT",          54070,  54070),
]
for crime, m1, spark_count in comparisons:
    match = "YES" if m1 == spark_count else "CLOSE"
    print(f"{crime:<25} {m1:>10} {spark_count:>12} {match:>6}")

print("\nConclusion: Spark DataFrame results match M1 MapReduce exactly.")
print("Same data, same computation, different execution engine.")

=== Task 1: Top 10 Crime Types (Spark DataFrame) ===


[Stage 33:=============================>                            (1 + 1) / 2]

+-------------------+------+
|        PrimaryType| count|
+-------------------+------+
|              THEFT|162688|
|            BATTERY|151930|
|    CRIMINAL DAMAGE| 91241|
|          NARCOTICS| 74127|
|            ASSAULT| 54070|
|MOTOR VEHICLE THEFT| 48494|
|           BURGLARY| 39872|
|      OTHER OFFENSE| 36893|
|            ROBBERY| 30991|
| DECEPTIVE PRACTICE| 30396|
+-------------------+------+
only showing top 10 rows


=== M1 MapReduce Results (for comparison) ===
Crime Type                  M1 Count  Spark Count  Match
---------------------------------------------------------
THEFT                         162688       162688    YES
BATTERY                       151930       151930    YES
CRIMINAL DAMAGE                91241        91241    YES
NARCOTICS                      74127        74127    YES
ASSAULT                        54070        54070    YES

Conclusion: Spark DataFrame results match M1 MapReduce exactly.
Same data, same computation, different execution engine.

In [14]:
# ============================================
# Task 2: Location Hotspots using Spark SQL
# Author: Dana Ghassan (ID: 231435)
# ============================================

print("=== Task 2: Location Hotspots (Spark SQL) ===")

# Register raw_df (has all original columns including Location Description)
raw_df.createOrReplaceTempView("crimes_raw")

location_hotspots = spark.sql("""
    SELECT `Location Description` as Location, COUNT(*) as total
    FROM crimes_raw
    GROUP BY `Location Description`
    ORDER BY total DESC
    LIMIT 10
""")

location_hotspots.show()

print("\n=== M1 MapReduce Results (for comparison) ===")
print(f"{'Location':<30} {'M1 Count':>10} {'Spark Count':>12} {'Diff':>6}")
print("-" * 62)
comparisons = [
    ("STREET",    245437, 248326),
    ("RESIDENCE", 136238, 136393),
    ("APARTMENT",  60925,  61235),
    ("SIDEWALK",   47407,  47506),
    ("OTHER",      29213,  29671),
]
for loc, m1, spark_count in comparisons:
    diff = spark_count - m1
    print(f"{loc:<30} {m1:>10} {spark_count:>12} {diff:>+6}")

print("\nConclusion: Minor differences (<1%) are expected.")
print("Likely due to rows dropped in M1 vs Spark's dropna() handling.")
print("Top locations and ranking are identical across both methods.")

=== Task 2: Location Hotspots (Spark SQL) ===


[Stage 36:=============================>                            (1 + 1) / 2]

+--------------------+------+
|            Location| total|
+--------------------+------+
|              STREET|248326|
|           RESIDENCE|136393|
|           APARTMENT| 61235|
|            SIDEWALK| 47506|
|               OTHER| 29671|
|PARKING LOT/GARAG...| 22436|
|               ALLEY| 18349|
|SCHOOL, PUBLIC, B...| 15776|
|    RESIDENCE-GARAGE| 14291|
|  SMALL RETAIL STORE| 13804|
+--------------------+------+


=== M1 MapReduce Results (for comparison) ===
Location                         M1 Count  Spark Count   Diff
--------------------------------------------------------------
STREET                             245437       248326  +2889
RESIDENCE                          136238       136393   +155
APARTMENT                           60925        61235   +310
SIDEWALK                            47407        47506    +99
OTHER                               29213        29671   +458

Conclusion: Minor differences (<1%) are expected.
Likely due to rows dropped in M1 vs Spark's dr

In [15]:
# ============================================
# Task 3: Crime Trend Over Years
# Author: Sema Raslan (ID: 231476)
# ============================================

from pyspark.sql.functions import col

print("=== Task 3: Crime Trend Over Years ===")

# Group by Year
yearly_df = df.groupBy("Year").count().orderBy("Year")

# Always show table (required for both modes)
yearly_df.show()

# Convert to pandas ONLY in local mode for plotting
if ENV == "local":
    try:
        import matplotlib
        matplotlib.use('Agg')  # Safe backend (no GUI needed)
        import matplotlib.pyplot as plt
        import os

        # Convert to pandas
        yearly_pd = yearly_df.toPandas()

        # Create plot
        plt.figure()
        plt.plot(yearly_pd["Year"], yearly_pd["count"])
        plt.xlabel("Year")
        plt.ylabel("Crime Count")
        plt.title("Crime Trend Over Years")

        # Save output
        os.makedirs("output", exist_ok=True)
        plt.savefig("output/crime_trend.png")

        print("Plot saved to: output/crime_trend.png")

    except Exception as e:
        print("Plotting skipped due to error:", e)

else:
    print("Cluster mode: Visualization skipped (table shown above).")

=== Task 3: Crime Trend Over Years ===


[Stage 39:=============================>                            (1 + 1) / 2]

+----+------+
|Year| count|
+----+------+
|2001|467301|
|2002|205266|
|2003|   985|
|2004|   915|
|2005|  1031|
|2006|   796|
|2007|   762|
|2008|  1010|
|2009|   910|
|2010|   695|
|2011|   770|
|2012|   800|
|2013|   714|
|2014|   825|
|2015|  1105|
|2016|  1339|
|2017|  1387|
|2018|  1327|
|2019|  1174|
|2020|  1832|
+----+------+
only showing top 20 rows

Cluster mode: Visualization skipped (table shown above).


In [16]:
# ============================================
# Task 4: Arrest Rate Analysis
# Author: Sema Raslan (ID: 231476)
# ============================================

from pyspark.sql.functions import col, avg, count, round

print("=== Task 4: Overall Arrest Statistics ===")

# Count True vs False (matches M1 exactly)
arrest_counts = df.groupBy("label") \
    .count() \
    .orderBy("label")

arrest_counts.show()

print("\n=== Overall Arrest Rate ===")

overall_rate = df.select(
    count("*").alias("total_cases"),
    round(avg(col("label")) * 100, 2).alias("arrest_rate_pct")
)

overall_rate.show()

# --------------------------------------------

print("\n=== Arrest Rate by Crime Type (Top 10 Highest) ===")

arrest_by_type = df.groupBy("PrimaryType") \
    .agg(
        count("*").alias("total_cases"),
        round(avg(col("label")) * 100, 2).alias("arrest_rate_pct")
    )

arrest_by_type.orderBy(col("arrest_rate_pct").desc()) \
    .show(10, truncate=False)

print("\n=== Arrest Rate by Crime Type (Bottom 5 Lowest) ===")

arrest_by_type.orderBy(col("arrest_rate_pct").asc()) \
    .show(5, truncate=False)

=== Task 4: Overall Arrest Statistics ===


+-----+------+
|label| count|
+-----+------+
|    0|571140|
|    1|221932|
+-----+------+


=== Overall Arrest Rate ===


+-----------+---------------+
|total_cases|arrest_rate_pct|
+-----------+---------------+
|     793072|          27.98|
+-----------+---------------+


=== Arrest Rate by Crime Type (Top 10 Highest) ===


+---------------------------------+-----------+---------------+
|PrimaryType                      |total_cases|arrest_rate_pct|
+---------------------------------+-----------+---------------+
|DOMESTIC VIOLENCE                |1          |100.0          |
|PUBLIC INDECENCY                 |17         |100.0          |
|NARCOTICS                        |74127      |99.88          |
|PROSTITUTION                     |9100       |99.88          |
|LIQUOR LAW VIOLATION             |2349       |99.83          |
|GAMBLING                         |1314       |99.77          |
|CONCEALED CARRY LICENSE VIOLATION|77         |94.81          |
|OTHER NARCOTIC VIOLATION         |11         |90.91          |
|INTERFERENCE WITH PUBLIC OFFICER |803        |80.7           |
|WEAPONS VIOLATION                |8893       |74.6           |
+---------------------------------+-----------+---------------+
only showing top 10 rows


=== Arrest Rate by Crime Type (Bottom 5 Lowest) ===


[Stage 51:=============================>                            (1 + 1) / 2]

+---------------+-----------+---------------+
|PrimaryType    |total_cases|arrest_rate_pct|
+---------------+-----------+---------------+
|NON-CRIMINAL   |1          |0.0            |
|INTIMIDATION   |92         |3.26           |
|BURGLARY       |39872      |6.74           |
|CRIMINAL DAMAGE|91241      |7.65           |
|ROBBERY        |30991      |9.83           |
+---------------+-----------+---------------+
only showing top 5 rows

